# UFC Main Cards in the 2020s

## Fight-by-fight analysis of numbered UFC events, 2020 to present

This notebook builds a dataset containing **main-card bouts from numbered UFC events only**. It excludes Fight Nights, prelims, early prelims, Contender Series events, and other promotions.

### NOTE:
This data does not include UFC Freedom 250 but I did include a place to insert that data in section 2.

### What the notebook produces

- One event table and one bout table
- Two fighter-centric records per bout
- Main-card appearances, wins, win percentage, and five-round-bout summaries
- Point-in-time Elo ratings and expected win probabilities
- Year and weight-class trends
- Interactive Altair charts
- Dated CSV, HTML, and optional PNG exports

### Data sources

- **ESPN public UFC event feed:** event identity, dates, bout order/start blocks, fighters, results, weight classes, records, and countries
- **Optional UFCStats/Kaggle CSV:** detailed striking, grappling, method, and control-time enrichment

## 1. Setup

Run the installation cell once if the packages are not already available, then restart the kernel.

In [3]:
from pathlib import Path
from datetime import date
import json
import re
import time

import numpy as np
import pandas as pd
import requests
import altair as alt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
alt.data_transformers.disable_max_rows()

print("Let's fight!")

Let's fight!


## 2. Configuration

`SPECIAL_NUMBERED_EVENTS` handles numbered events whose names do not match the standard `UFC 246` pattern. Add an event here only after verifying that it belongs in the numbered-event series.

In [5]:
START_YEAR = 2020
END_YEAR = pd.Timestamp.today().year
RUN_DATE = date.today().isoformat()

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = Path("outputs") / RUN_DATE
TABLE_DIR = OUTPUT_DIR / "tables"
CHART_DIR = OUTPUT_DIR / "charts"

for folder in [RAW_DIR, PROCESSED_DIR, TABLE_DIR, CHART_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ESPN_SCOREBOARD_URL = (
    "https://site.api.espn.com/apis/site/v2/sports/mma/ufc/scoreboard"
)

NUMBERED_EVENT_PATTERN = re.compile(r"^UFC\s+\d+\b", flags=re.IGNORECASE)

SPECIAL_NUMBERED_EVENTS = {
    # Add verified special numbered-event names here, for example:
    # "UFC Freedom 250: Fighter A vs. Fighter B",
}

# Validation only. Main cards outside this range remain in the dataset but are flagged.
EXPECTED_MAIN_CARD_MIN = 4
EXPECTED_MAIN_CARD_MAX = 6

print(f"Analysis period: {START_YEAR}–{END_YEAR}")
print(f"Output folder: {OUTPUT_DIR}")

Analysis period: 2020–2026
Output folder: outputs/2026-08-18


## 3. Download the ESPN event feed

The download is cached by year. Set `REFRESH_ESPN = True` after a newly completed numbered event; otherwise the notebook reuses the saved JSON files.

In [7]:
REFRESH_ESPN = False

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; UFC-2020s-analytics/1.0)"
})


def fetch_espn_year(year, refresh=False, pause=0.25):
    cache_file = RAW_DIR / f"espn_ufc_scoreboard_{year}.json"

    if cache_file.exists() and not refresh:
        return json.loads(cache_file.read_text(encoding="utf-8"))

    response = session.get(
        ESPN_SCOREBOARD_URL,
        params={"dates": str(year)},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    cache_file.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    time.sleep(pause)
    return payload


year_payloads = {}
download_errors = []

for year in range(START_YEAR, END_YEAR + 1):
    try:
        year_payloads[year] = fetch_espn_year(year, refresh=REFRESH_ESPN)
        print(year, "events:", len(year_payloads[year].get("events", [])))
    except Exception as exc:
        download_errors.append({"year": year, "error": str(exc)})
        print(f"Could not download {year}: {exc}")

if not year_payloads:
    raise RuntimeError(
        "No ESPN data was available. Check the internet connection and rerun this cell."
    )

if download_errors:
    display(pd.DataFrame(download_errors))

2020 events: 53
2021 events: 57
2022 events: 56
2023 events: 57
2024 events: 52
2025 events: 52
2026 events: 47


## 4. Numbered events and identify main-card bouts only

In [9]:
def is_numbered_event(event_name):
    name = str(event_name).strip()
    return bool(NUMBERED_EVENT_PATTERN.search(name)) or name in SPECIAL_NUMBERED_EVENTS


def record_summary(competitor):
    records = competitor.get("records") or []
    overall = next((r for r in records if r.get("type") == "total"), None)
    return (overall or {}).get("summary")


def fighter_country(competitor):
    return ((competitor.get("athlete") or {}).get("flag") or {}).get("alt")


def parse_event(event):
    event_name = event.get("name", "")
    if not is_numbered_event(event_name):
        return [], []

    event_date = pd.to_datetime(event.get("date"), utc=True, errors="coerce")
    competitions = event.get("competitions") or []
    completed = [
        c for c in competitions
        if ((c.get("status") or {}).get("type") or {}).get("completed") is True
    ]
    if not completed:
        return [], []

    starts = pd.to_datetime(
        [c.get("startDate") or c.get("date") for c in completed],
        utc=True,
        errors="coerce",
    )
    valid_starts = [x for x in starts if not pd.isna(x)]
    main_card_start = max(valid_starts) if valid_starts else pd.NaT

    venue = event.get("venues") or []
    venue = venue[0] if venue else (completed[0].get("venue") or {})
    address = venue.get("address") or {}

    event_row = {
        "event_id": str(event.get("id")),
        "event_name": event_name,
        "event_date": event_date,
        "year": event_date.year if not pd.isna(event_date) else np.nan,
        "venue": venue.get("fullName"),
        "city": address.get("city"),
        "state": address.get("state"),
        "country": address.get("country"),
        "total_completed_bouts": len(completed),
    }

    bout_rows = []
    for source_index, competition in enumerate(completed):
        bout_start = pd.to_datetime(
            competition.get("startDate") or competition.get("date"),
            utc=True,
            errors="coerce",
        )
        is_main_card = bool(
            not pd.isna(main_card_start)
            and not pd.isna(bout_start)
            and bout_start == main_card_start
        )

        competitors = competition.get("competitors") or []
        if len(competitors) != 2:
            continue

        winner = next((c for c in competitors if c.get("winner") is True), None)
        loser = next((c for c in competitors if c.get("winner") is False), None)
        status = competition.get("status") or {}
        status_type = status.get("type") or {}
        regulation = (competition.get("format") or {}).get("regulation") or {}

        bout_rows.append({
            "event_id": str(event.get("id")),
            "event_name": event_name,
            "event_date": event_date,
            "year": event_date.year if not pd.isna(event_date) else np.nan,
            "bout_id": str(competition.get("id")),
            "source_order": source_index + 1,
            "bout_start_utc": bout_start,
            "main_card_start_utc": main_card_start,
            "card_section": "Main Card" if is_main_card else "Prelims",
            "broadcast": competition.get("broadcast"),
            "weight_class": (competition.get("type") or {}).get("abbreviation"),
            "scheduled_rounds": regulation.get("periods"),
            "completed_round": status.get("period"),
            "ending_clock": status.get("displayClock"),
            "status": status_type.get("description"),
            "winner_id": str(
                (winner or {}).get("id")
                or ((winner or {}).get("athlete") or {}).get("id")
                or ""
            ),
            "winner": ((winner or {}).get("athlete") or {}).get("displayName"),
            "winner_country": fighter_country(winner or {}),
            "winner_record": record_summary(winner or {}),
            "loser_id": str(
                (loser or {}).get("id")
                or ((loser or {}).get("athlete") or {}).get("id")
                or ""
            ),
            "loser": ((loser or {}).get("athlete") or {}).get("displayName"),
            "loser_country": fighter_country(loser or {}),
            "loser_record": record_summary(loser or {}),
        })

    return [event_row], bout_rows


event_rows, bout_rows = [], []
for payload in year_payloads.values():
    for event in payload.get("events", []):
        new_events, new_bouts = parse_event(event)
        event_rows.extend(new_events)
        bout_rows.extend(new_bouts)

events = (
    pd.DataFrame(event_rows)
    .drop_duplicates("event_id")
    .sort_values("event_date")
    .reset_index(drop=True)
)

all_numbered_bouts = (
    pd.DataFrame(bout_rows)
    .drop_duplicates("bout_id")
    .sort_values(["event_date", "bout_start_utc", "source_order"])
    .reset_index(drop=True)
)

main_card_bouts = (
    all_numbered_bouts.loc[all_numbered_bouts["card_section"].eq("Main Card")]
    .copy()
    .sort_values(["event_date", "source_order"])
    .reset_index(drop=True)
)

print("Numbered events:", len(events))
print("All completed bouts at those events:", len(all_numbered_bouts))
print("Main-card bouts:", len(main_card_bouts))
display(main_card_bouts.tail(10))

Numbered events: 85
All completed bouts at those events: 1060
Main-card bouts: 427


,event_id,event_name,event_date,year,bout_id,source_order,bout_start_utc,main_card_start_utc,card_section,broadcast,weight_class,scheduled_rounds,completed_round,ending_clock,status,winner_id,winner,winner_country,winner_record,loser_id,loser,loser_country,loser_record
417,600059148,UFC 329: McGregor vs. Holloway 2,2026-07-11 21:00:00+00:00,2026,401881161,10,2026-07-12 01:00:00+00:00,2026-07-12 01:00:00+00:00,Main Card,Paramount+,Lightweight,3,1,4:59,Final,2502364,King Green,USA,36-17-1,4425604,Terrance McKinney,USA,18-9-0
418,600059148,UFC 329: McGregor vs. Holloway 2,2026-07-11 21:00:00+00:00,2026,401873376,11,2026-07-12 01:00:00+00:00,2026-07-12 01:00:00+00:00,Main Card,Paramount+,Flyweight,3,3,3:40,Final,4239928,Brandon Royval,USA,18-9-0,5147737,Lone'er Kavanagh,England,10-2-0
419,600059148,UFC 329: McGregor vs. Holloway 2,2026-07-11 21:00:00+00:00,2026,401873375,12,2026-07-12 01:00:00+00:00,2026-07-12 01:00:00+00:00,Main Card,Paramount+,Bantamweight,3,3,5:00,Final,4410868,Mario Bautista,USA,18-3-0,4294504,Cory Sandhagen,USA,18-7-0
420,600059148,UFC 329: McGregor vs. Holloway 2,2026-07-11 21:00:00+00:00,2026,401873374,13,2026-07-12 01:00:00+00:00,2026-07-12 01:00:00+00:00,Main Card,Paramount+,Lightweight,3,1,0:52,Final,4008549,Paddy Pimblett,England,24-4-0,4895362,Benoît Saint Denis,France,17-4-0
421,600059148,UFC 329: McGregor vs. Holloway 2,2026-07-11 21:00:00+00:00,2026,401867788,14,2026-07-12 01:00:00+00:00,2026-07-12 01:00:00+00:00,Main Card,Paramount+,Welterweight,5,1,1:09,Final,2614933,Max Holloway,USA,28-9-0,3022677,Conor McGregor,Ireland,22-7-0
422,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401879330,8,2026-08-16 01:00:00+00:00,2026-08-16 01:00:00+00:00,Main Card,Paramount+,Lightweight,3,2,1:32,Final,5074131,Esteban Ribovics,Argentina,16-3-0,2526299,Edson Barboza,Brazil,24-15-0
423,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401881928,9,2026-08-16 01:00:00+00:00,2026-08-16 01:00:00+00:00,Main Card,Paramount+,Middleweight,3,2,4:25,Final,4685871,Dustin Stoltzfus,USA,17-8-0,5211221,Mansur Abdul-Malik,USA,9-2-1
424,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401886764,10,2026-08-16 01:00:00+00:00,2026-08-16 01:00:00+00:00,Main Card,Paramount+,Lightweight,3,1,0:39,Final,4339490,Jalin Turner,USA,16-9-0,5172124,Kauê Fernandes,Brazil,11-3-0
425,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401878072,11,2026-08-16 01:00:00+00:00,2026-08-16 01:00:00+00:00,Main Card,Paramount+,W Strawweight,5,5,5:00,Final,4021217,Mackenzie Dern,Brazil,17-5-0,4089026,Gillian Robertson,Canada,17-9-0
426,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401869336,12,2026-08-16 01:00:00+00:00,2026-08-16 01:00:00+00:00,Main Card,Paramount+,Welterweight,5,5,5:00,Final,3332412,Islam Makhachev,Russia,29-1-0,4738092,Ian Machado Garry,Ireland,17-2-0


## 5. Validate the main-card classification

Review every row marked `needs_review`. Reasons include unusual broadcast schedules, cancelled fights, rescheduled bouts, or incomplete source timing. Add corrections to `data/card_overrides.csv` rather than changing the raw data.

The optional override file should contain:

```text
bout_id,card_section
123456,Main Card
123457,Prelims
```

In [11]:
card_validation = (
    main_card_bouts.groupby(
        ["event_id", "event_name", "event_date"], as_index=False
    )
    .agg(
        main_card_bouts=("bout_id", "nunique"),
        main_card_broadcasts=("broadcast", lambda x: ", ".join(sorted(set(x.dropna())))),
        main_card_start=("main_card_start_utc", "first"),
    )
)

card_validation["needs_review"] = ~card_validation["main_card_bouts"].between(
    EXPECTED_MAIN_CARD_MIN, EXPECTED_MAIN_CARD_MAX
)

display(card_validation.loc[card_validation["needs_review"]])

OVERRIDE_FILE = DATA_DIR / "card_overrides.csv"

if OVERRIDE_FILE.exists():
    overrides = pd.read_csv(OVERRIDE_FILE, dtype={"bout_id": "string"})
    required = {"bout_id", "card_section"}
    if not required.issubset(overrides.columns):
        raise ValueError(f"{OVERRIDE_FILE} must contain {sorted(required)}")

    overrides["bout_id"] = overrides["bout_id"].astype(str)
    all_numbered_bouts["bout_id"] = all_numbered_bouts["bout_id"].astype(str)
    mapping = overrides.set_index("bout_id")["card_section"]
    all_numbered_bouts["card_section"] = (
        all_numbered_bouts["bout_id"].map(mapping)
        .fillna(all_numbered_bouts["card_section"])
    )
    main_card_bouts = all_numbered_bouts.loc[
        all_numbered_bouts["card_section"].eq("Main Card")
    ].copy()
    print("Applied", len(overrides), "card override(s).")
else:
    print("No card override file found. Automated classification retained.")

,event_id,event_name,event_date,main_card_bouts,main_card_broadcasts,main_card_start,needs_review


No card override file found. Automated classification retained.


## 6. Create dataset

Each bout produces one row from the winner’s perspective and one from the loser’s perspective. Draws and no contests are kept only if both competitors are identified.

In [13]:
def fight_seconds(completed_round, ending_clock, scheduled_rounds):
    try:
        round_number = int(completed_round)
        minutes, seconds = map(int, str(ending_clock).split(":"))
        # ESPN commonly reports elapsed time in the final round.
        return (round_number - 1) * 300 + minutes * 60 + seconds
    except (TypeError, ValueError, AttributeError):
        return np.nan


def make_fighter_rows(row):
    shared = {
        "event_id": row.event_id,
        "event_name": row.event_name,
        "event_date": row.event_date,
        "year": row.year,
        "bout_id": row.bout_id,
        "card_section": row.card_section,
        "broadcast": row.broadcast,
        "weight_class": row.weight_class,
        "scheduled_rounds": row.scheduled_rounds,
        "five_round_bout": row.scheduled_rounds == 5,
        "completed_round": row.completed_round,
        "ending_clock": row.ending_clock,
        "fight_seconds": fight_seconds(
            row.completed_round, row.ending_clock, row.scheduled_rounds
        ),
    }
    return [
        {
            **shared,
            "fighter_id": row.winner_id,
            "fighter": row.winner,
            "fighter_country": row.winner_country,
            "fighter_record_at_event": row.winner_record,
            "opponent_id": row.loser_id,
            "opponent": row.loser,
            "opponent_country": row.loser_country,
            "opponent_record_at_event": row.loser_record,
            "result": "W",
            "win": 1,
        },
        {
            **shared,
            "fighter_id": row.loser_id,
            "fighter": row.loser,
            "fighter_country": row.loser_country,
            "fighter_record_at_event": row.loser_record,
            "opponent_id": row.winner_id,
            "opponent": row.winner,
            "opponent_country": row.winner_country,
            "opponent_record_at_event": row.winner_record,
            "result": "L",
            "win": 0,
        },
    ]


fighter_rows = []
for row in main_card_bouts.itertuples(index=False):
    if pd.notna(row.winner) and pd.notna(row.loser):
        fighter_rows.extend(make_fighter_rows(row))

fighter_fights = (
    pd.DataFrame(fighter_rows)
    .sort_values(["event_date", "bout_id", "result"], ascending=[True, True, False])
    .reset_index(drop=True)
)

assert not fighter_fights.duplicated(["bout_id", "fighter_id"]).any()
assert set(fighter_fights["card_section"].unique()) == {"Main Card"}
assert len(fighter_fights) == 2 * fighter_fights["bout_id"].nunique()

print("Fighter-centric records:", len(fighter_fights))
print("Unique fighters:", fighter_fights["fighter_id"].nunique())
display(fighter_fights.tail(10))

Fighter-centric records: 836
Unique fighters: 332


,event_id,event_name,event_date,year,bout_id,card_section,broadcast,weight_class,scheduled_rounds,five_round_bout,completed_round,ending_clock,fight_seconds,fighter_id,fighter,fighter_country,fighter_record_at_event,opponent_id,opponent,opponent_country,opponent_record_at_event,result,win
826,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401869336,Main Card,Paramount+,Welterweight,5,True,5,5:00,1500,3332412,Islam Makhachev,Russia,29-1-0,4738092,Ian Machado Garry,Ireland,17-2-0,W,1
827,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401869336,Main Card,Paramount+,Welterweight,5,True,5,5:00,1500,4738092,Ian Machado Garry,Ireland,17-2-0,3332412,Islam Makhachev,Russia,29-1-0,L,0
828,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401878072,Main Card,Paramount+,W Strawweight,5,True,5,5:00,1500,4021217,Mackenzie Dern,Brazil,17-5-0,4089026,Gillian Robertson,Canada,17-9-0,W,1
829,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401878072,Main Card,Paramount+,W Strawweight,5,True,5,5:00,1500,4089026,Gillian Robertson,Canada,17-9-0,4021217,Mackenzie Dern,Brazil,17-5-0,L,0
830,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401879330,Main Card,Paramount+,Lightweight,3,False,2,1:32,392,5074131,Esteban Ribovics,Argentina,16-3-0,2526299,Edson Barboza,Brazil,24-15-0,W,1
831,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401879330,Main Card,Paramount+,Lightweight,3,False,2,1:32,392,2526299,Edson Barboza,Brazil,24-15-0,5074131,Esteban Ribovics,Argentina,16-3-0,L,0
832,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401881928,Main Card,Paramount+,Middleweight,3,False,2,4:25,565,4685871,Dustin Stoltzfus,USA,17-8-0,5211221,Mansur Abdul-Malik,USA,9-2-1,W,1
833,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401881928,Main Card,Paramount+,Middleweight,3,False,2,4:25,565,5211221,Mansur Abdul-Malik,USA,9-2-1,4685871,Dustin Stoltzfus,USA,17-8-0,L,0
834,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401886764,Main Card,Paramount+,Lightweight,3,False,1,0:39,39,4339490,Jalin Turner,USA,16-9-0,5172124,Kauê Fernandes,Brazil,11-3-0,W,1
835,600059185,UFC 330: Makhachev vs. Machado Garry,2026-08-15 21:30:00+00:00,2026,401886764,Main Card,Paramount+,Lightweight,3,False,1,0:39,39,5172124,Kauê Fernandes,Brazil,11-3-0,4339490,Jalin Turner,USA,16-9-0,L,0


## 7. Point-in-time Elo ratings

Elo is calculated chronologically and uses only information available before each bout. Every fighter begins at 1500. The model is an analytical benchmark—not a calibrated betting model. The Elo rating estimates a fighter’s strength based on previous results and opponent quality. Ratings rise after wins and fall after losses, with larger changes for unexpected outcomes. “Point-in-time” means each rating uses only information available before that fight, without considering future results. These rating are limited as they do not include pre-2020 bouts. I just thought this was a cool thing to show.

In [15]:
ELO_START = 1500.0
ELO_K = 32.0


def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10.0 ** ((rating_b - rating_a) / 400.0))


ratings = {}
elo_bouts = []

ordered_bouts = main_card_bouts.sort_values(
    ["event_date", "bout_start_utc", "source_order"]
)

for bout in ordered_bouts.itertuples(index=False):
    if not bout.winner_id or not bout.loser_id:
        continue

    winner_pre = ratings.get(bout.winner_id, ELO_START)
    loser_pre = ratings.get(bout.loser_id, ELO_START)
    winner_expected = expected_score(winner_pre, loser_pre)
    loser_expected = 1.0 - winner_expected

    winner_post = winner_pre + ELO_K * (1.0 - winner_expected)
    loser_post = loser_pre + ELO_K * (0.0 - loser_expected)

    ratings[bout.winner_id] = winner_post
    ratings[bout.loser_id] = loser_post

    elo_bouts.extend([
        {
            "bout_id": bout.bout_id,
            "fighter_id": bout.winner_id,
            "elo_pre": winner_pre,
            "opponent_elo_pre": loser_pre,
            "expected_win_probability": winner_expected,
            "elo_post": winner_post,
            "elo_change": winner_post - winner_pre,
        },
        {
            "bout_id": bout.bout_id,
            "fighter_id": bout.loser_id,
            "elo_pre": loser_pre,
            "opponent_elo_pre": winner_pre,
            "expected_win_probability": loser_expected,
            "elo_post": loser_post,
            "elo_change": loser_post - loser_pre,
        },
    ])

elo_history = pd.DataFrame(elo_bouts)
fighter_fights = fighter_fights.merge(
    elo_history,
    on=["bout_id", "fighter_id"],
    how="left",
    validate="one_to_one",
)

fighter_fights["upset_win"] = (
    fighter_fights["win"].eq(1)
    & fighter_fights["expected_win_probability"].lt(0.50)
)

display(
    fighter_fights.loc[fighter_fights["upset_win"]]
    .sort_values("expected_win_probability")
    [[
        "event_date", "event_name", "fighter", "opponent",
        "expected_win_probability", "elo_pre", "opponent_elo_pre"
    ]]
    .head(15)
)

,event_date,event_name,fighter,opponent,expected_win_probability,elo_pre,opponent_elo_pre
806,2026-05-09 21:00:00+00:00,UFC 328: Chimaev vs. Strickland,Sean Strickland,Khamzat Chimaev,0.360014,1491.024945,1590.965093
758,2025-12-06 23:00:00+00:00,UFC 323: Dvalishvili vs. Yan 2,Petr Yan,Merab Dvalishvili,0.360061,1503.475126,1603.380068
464,2023-09-09 22:00:00+00:00,UFC 293: Adesanya vs. Strickland,Sean Strickland,Israel Adesanya,0.396640,1484.000000,1556.872181
760,2025-12-06 23:00:00+00:00,UFC 323: Dvalishvili vs. Yan 2,Joshua Van,Alexandre Pantoja,0.399219,1516.033980,1587.035768
540,2024-04-13 22:00:00+00:00,UFC 300: Pereira vs. Hill,Max Holloway,Justin Gaethje,0.410908,1470.817596,1533.393257
536,2024-04-13 22:00:00+00:00,UFC 300: Pereira vs. Hill,Arman Tsarukyan,Charles Oliveira,0.413915,1500.000000,1560.419482
746,2025-10-25 14:00:00+00:00,UFC 321: Aspinall vs. Gane,Umar Nurmagomedov,Mario Bautista,0.415890,1486.914118,1545.920333
390,2023-03-04 22:30:00+00:00,UFC 285: Jones vs. Gane,Alexa Grasso,Valentina Shevchenko,0.418645,1516.000000,1573.038066
632,2024-12-07 23:00:00+00:00,UFC 310: Pantoja vs. Asakura,Ciryl Gane,Alexander Volkov,0.431409,1499.331434,1547.295968
834,2026-08-15 21:30:00+00:00,UFC 330: Makhachev vs. Machado Garry,Jalin Turner,Kauê Fernandes,0.432747,1452.982737,1500.000000


## 8. Player summaries

The minimum-fight filter prevents a fighter with one appearance from automatically topping the win-percentage table.

In [17]:
MIN_FIGHTS_FOR_RATE = 3

fighter_summary = (
    fighter_fights.groupby(["fighter_id", "fighter"], as_index=False)
    .agg(
        main_card_appearances=("bout_id", "nunique"),
        wins=("win", "sum"),
        five_round_bouts=("five_round_bout", "sum"),
        upsets=("upset_win", "sum"),
        first_appearance=("event_date", "min"),
        latest_appearance=("event_date", "max"),
        current_elo=("elo_post", "last"),
        peak_postfight_elo=("elo_post", "max"),
    )
)

fighter_summary["losses"] = (
    fighter_summary["main_card_appearances"] - fighter_summary["wins"]
)
fighter_summary["win_percentage"] = (
    100 * fighter_summary["wins"] / fighter_summary["main_card_appearances"]
)

fighter_summary = fighter_summary.sort_values(
    ["wins", "win_percentage", "main_card_appearances"],
    ascending=False,
).reset_index(drop=True)

display(fighter_summary.head(20))

,fighter_id,fighter,main_card_appearances,wins,five_round_bouts,upsets,first_appearance,latest_appearance,current_elo,peak_postfight_elo,losses,win_percentage
0,3332412,Islam Makhachev,9,9,7,2,2021-03-06 22:15:00+00:00,2026-08-15 21:30:00+00:00,1631.638532,1631.638532,0,100.000000
1,2554705,Valentina Shevchenko,9,8,9,0,2020-02-08 23:30:00+00:00,2025-11-15 23:00:00+00:00,1598.155368,1598.155368,1,88.888889
2,4705658,Alex Pereira,10,8,8,2,2022-07-02 22:00:00+00:00,2025-10-04 22:00:00+00:00,1587.923568,1588.883568,2,80.000000
3,4205093,Sean O'Malley,11,8,4,2,2020-06-06 22:00:00+00:00,2026-01-24 22:30:00+00:00,1569.996260,1590.468153,3,72.727273
4,3948572,Merab Dvalishvili,8,7,5,1,2020-08-15 22:00:00+00:00,2025-12-06 23:00:00+00:00,1582.902023,1603.380068,1,87.500000
5,2504169,Charles Oliveira,10,7,7,3,2020-12-13 00:30:00+00:00,2026-03-07 22:30:00+00:00,1552.613079,1561.855925,3,70.000000
6,3949584,Alexander Volkanovski,10,7,10,1,2020-07-11 22:00:00+00:00,2026-01-31 22:00:00+00:00,1555.715550,1559.010224,3,70.000000
7,2560746,Alexandre Pantoja,7,6,6,1,2022-07-30 22:00:00+00:00,2025-12-06 23:00:00+00:00,1567.810785,1587.035768,1,85.714286
8,4684751,Khamzat Chimaev,7,6,4,2,2021-10-30 14:30:00+00:00,2026-05-09 21:00:00+00:00,1570.485554,1590.965093,1,85.714286
9,3022345,Justin Gaethje,9,6,6,3,2020-05-09 22:00:00+00:00,2026-01-24 22:30:00+00:00,1546.381730,1546.381730,3,66.666667


## 9. Altair visualizations

I prefer Altair over matplotlib because of Altair's ability to zoom in and out of graphs and charts.

In [19]:
def chart_title(text):
    return alt.TitleParams(
        text=text,
        subtitle="Numbered UFC events, main-card bouts only | 2020 to present",
        anchor="start",
    )

top_wins = fighter_summary.head(20).copy()
chart_wins = (
    alt.Chart(top_wins)
    .mark_bar(color="#C8102E")
    .encode(
        x=alt.X("wins:Q", title="Main-card wins"),
        y=alt.Y("fighter:N", sort="-x", title=None),
        tooltip=[
            "fighter:N", "wins:Q", "losses:Q", "main_card_appearances:Q",
            alt.Tooltip("win_percentage:Q", format=".1f"),
        ],
    )
    .properties(width=700, height=500, title=chart_title("Most main-card wins"))
)

eligible_rates = (
    fighter_summary.loc[
        fighter_summary["main_card_appearances"] >= MIN_FIGHTS_FOR_RATE
    ]
    .sort_values(["win_percentage", "wins"], ascending=False)
    .head(20)
)
chart_win_rate = (
    alt.Chart(eligible_rates)
    .mark_bar(color="#1F4E79")
    .encode(
        x=alt.X("win_percentage:Q", title="Win percentage", scale=alt.Scale(domain=[0, 100])),
        y=alt.Y("fighter:N", sort="-x", title=None),
        tooltip=[
            "fighter:N", alt.Tooltip("win_percentage:Q", format=".1f"),
            "wins:Q", "losses:Q", "main_card_appearances:Q",
        ],
    )
    .properties(width=700, height=500, title=chart_title(f"Best win percentage ({MIN_FIGHTS_FOR_RATE}+ appearances)"))
)

yearly = (
    main_card_bouts.groupby("year", as_index=False)
    .agg(main_card_bouts=("bout_id", "nunique"), events=("event_id", "nunique"))
)
chart_yearly = (
    alt.Chart(yearly)
    .mark_line(point=True, color="#C8102E", strokeWidth=3)
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("main_card_bouts:Q", title="Main-card bouts"),
        tooltip=["year:O", "main_card_bouts:Q", "events:Q"],
    )
    .properties(width=700, height=350, title=chart_title("Main-card bouts by year"))
)

division = (
    main_card_bouts.groupby("weight_class", as_index=False)
    .agg(bouts=("bout_id", "nunique"))
    .sort_values("bouts", ascending=False)
)
chart_division = (
    alt.Chart(division)
    .mark_bar(color="#4A4A4A")
    .encode(
        x=alt.X("bouts:Q", title="Main-card bouts"),
        y=alt.Y("weight_class:N", sort="-x", title=None),
        tooltip=["weight_class:N", "bouts:Q"],
    )
    .properties(width=700, height=400, title=chart_title("Main-card representation by weight class"))
)

top_elo_names = set(
    fighter_summary.nlargest(15, "current_elo")["fighter_id"].astype(str)
)
elo_plot = fighter_fights.loc[
    fighter_fights["fighter_id"].astype(str).isin(top_elo_names)
].copy()
chart_elo = (
    alt.Chart(elo_plot)
    .mark_line(point=True)
    .encode(
        x=alt.X("event_date:T", title="Event date"),
        y=alt.Y("elo_post:Q", title="Post-fight Elo", scale=alt.Scale(zero=False)),
        color=alt.Color("fighter:N", title="Fighter"),
        tooltip=[
            "event_date:T", "event_name:N", "fighter:N", "opponent:N",
            "result:N", alt.Tooltip("elo_post:Q", format=".1f"),
        ],
    )
    .properties(width=800, height=450, title=chart_title("Elo paths of current leaders"))
    .interactive()
)

upsets = (
    fighter_fights.loc[fighter_fights["upset_win"]]
    .nsmallest(20, "expected_win_probability")
    .copy()
)
upsets["matchup"] = upsets["fighter"] + " def. " + upsets["opponent"]
chart_upsets = (
    alt.Chart(upsets)
    .mark_bar(color="#D4A72C")
    .encode(
        x=alt.X("expected_win_probability:Q", title="Pre-fight expected win probability", axis=alt.Axis(format="%")),
        y=alt.Y("matchup:N", sort="x", title=None),
        tooltip=[
            "event_date:T", "event_name:N", "matchup:N",
            alt.Tooltip("expected_win_probability:Q", format=".1%"),
        ],
    )
    .properties(width=700, height=500, title=chart_title("Largest Elo upsets"))
)

charts = {
    "main_card_wins": chart_wins,
    "win_percentage": chart_win_rate,
    "bouts_by_year": chart_yearly,
    "weight_class_representation": chart_division,
    "elo_over_time": chart_elo,
    "largest_elo_upsets": chart_upsets,
}

chart_wins

alt.Chart(...)

In [20]:
# Display the remaining charts.
for name, chart in charts.items():
    display(Markdown(f"### {name.replace('_', ' ').title()}"))
    display(chart)

### Main Card Wins

alt.Chart(...)

### Win Percentage

alt.Chart(...)

### Bouts By Year

...

alt.Chart(...)

### Elo Over Time

alt.Chart(...)

### Largest Elo Upsets

alt.Chart(...)

## 10. Fighter and event drill-downs

Change the selections below to inspect an individual fighter or numbered event.

In [22]:
SELECTED_FIGHTER = "Paddy Pimblett"

fighter_drilldown = (
    fighter_fights.loc[fighter_fights["fighter"].eq(SELECTED_FIGHTER)]
    [[
        "event_date", "event_name", "opponent", "result", "weight_class",
        "five_round_bout", "expected_win_probability", "elo_pre", "elo_post"
    ]]
    .sort_values("event_date", ascending=False)
)

print(SELECTED_FIGHTER)
display(fighter_drilldown)

Paddy Pimblett


,event_date,event_name,opponent,result,weight_class,five_round_bout,expected_win_probability,elo_pre,elo_post
818,2026-07-11 21:00:00+00:00,UFC 329: McGregor vs. Holloway 2,Benoît Saint Denis,W,Lightweight,False,0.486197,1538.896517,1555.338220
767,2026-01-24 22:30:00+00:00,UFC 324: Gaethje vs. Pimblett,Justin Gaethje,L,Lightweight,True,0.538773,1556.137244,1538.896517
672,2025-04-12 22:00:00+00:00,UFC 314: Volkanovski vs. Lopes,Michael Chandler,W,Lightweight,True,0.604107,1543.468658,1556.137244
576,2024-07-27 22:00:00+00:00,UFC 304: Edwards vs. Muhammad 2,King Green,W,Lightweight,False,0.499265,1527.445136,1543.468658
498,2023-12-16 23:30:00+00:00,UFC 296: Edwards vs. Covington,Tony Ferguson,W,Lightweight,False,0.642339,1516.000000,1527.445136
358,2022-12-10 23:30:00+00:00,UFC 282: Blachowicz vs. Ankalaev,Jared Gordon,W,Lightweight,False,0.500000,1500.000000,1516.000000


In [23]:
SELECTED_EVENT = events.iloc[-1]["event_name"]

event_drilldown = main_card_bouts.loc[
    main_card_bouts["event_name"].eq(SELECTED_EVENT),
    [
        "event_date", "event_name", "winner", "loser", "weight_class",
        "scheduled_rounds", "completed_round", "ending_clock", "broadcast"
    ],
]

print(SELECTED_EVENT)
display(event_drilldown)

UFC 330: Makhachev vs. Machado Garry


,event_date,event_name,winner,loser,weight_class,scheduled_rounds,completed_round,ending_clock,broadcast
422,2026-08-15 21:30:00+00:00,UFC 330: Makhachev vs. Machado Garry,Esteban Ribovics,Edson Barboza,Lightweight,3,2,1:32,Paramount+
423,2026-08-15 21:30:00+00:00,UFC 330: Makhachev vs. Machado Garry,Dustin Stoltzfus,Mansur Abdul-Malik,Middleweight,3,2,4:25,Paramount+
424,2026-08-15 21:30:00+00:00,UFC 330: Makhachev vs. Machado Garry,Jalin Turner,Kauê Fernandes,Lightweight,3,1,0:39,Paramount+
425,2026-08-15 21:30:00+00:00,UFC 330: Makhachev vs. Machado Garry,Mackenzie Dern,Gillian Robertson,W Strawweight,5,5,5:00,Paramount+
426,2026-08-15 21:30:00+00:00,UFC 330: Makhachev vs. Machado Garry,Islam Makhachev,Ian Machado Garry,Welterweight,5,5,5:00,Paramount+


## 12. Export tables and charts

In [25]:
events.to_csv(TABLE_DIR / "numbered_ufc_events_2020s.csv", index=False)
main_card_bouts.to_csv(TABLE_DIR / "main_card_bouts_2020s.csv", index=False)
fighter_fights.to_csv(TABLE_DIR / "fighter_fights_2020s.csv", index=False)
fighter_summary.to_csv(TABLE_DIR / "fighter_summary_2020s.csv", index=False)
card_validation.to_csv(TABLE_DIR / "card_classification_validation.csv", index=False)

png_errors = []
for name, chart in charts.items():
    chart.save(CHART_DIR / f"{name}.html")
    try:
        chart.save(CHART_DIR / f"{name}.png", scale_factor=2)
    except Exception as exc:
        png_errors.append({"chart": name, "error": str(exc)})

print("Tables saved to:", TABLE_DIR)
print("Charts saved to:", CHART_DIR)
if png_errors:
    print("Some PNG files were not created. Install vl-convert-python and rerun.")
    display(pd.DataFrame(png_errors))

Tables saved to: outputs/2026-08-18/tables
Charts saved to: outputs/2026-08-18/charts
Some PNG files were not created. Install vl-convert-python and rerun.


,chart,error
0,main_card_wins,Saving charts in 'png' format requires the vl-...
1,win_percentage,Saving charts in 'png' format requires the vl-...
2,bouts_by_year,Saving charts in 'png' format requires the vl-...
3,weight_class_representation,Saving charts in 'png' format requires the vl-...
4,elo_over_time,Saving charts in 'png' format requires the vl-...
5,largest_elo_upsets,Saving charts in 'png' format requires the vl-...
